In [ ]:
import earthaccess
import os
import warnings
import csv
from osgeo import gdal
import numpy as np
import math
import rasterio as rio
import xarray as xr
import holoviews as hv
import hvplot.xarray
import netCDF4 as nc
#import cv2 as cv

import glob
import sys
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import Affine
from scipy import linalg
from lxml import etree
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation, rotate
from skimage.transform import hough_line, hough_line_peaks
from datetime import datetime, timedelta, timezone
from scipy.ndimage import distance_transform_edt
from skimage.restoration import denoise_tv_chambolle, inpaint_biharmonic
from scipy.ndimage import gaussian_filter

wav_min = 418
wav_max = 492
bin_size = 1
polydeg = 3
sys.path.append('../')
from config import CONFIG, CROSS_SECTIONS, LOCS
from REFERENCE_PLANTS import REFERENCE_PLANTS

sys.path.append('../../EMIT-Data-Resources/python/modules/')
from emit_tools import emit_xarray, ortho_xr
help(emit_xarray)

sys.path.append('../../datasets/')
from get_geosfp import get_geosfp_wind
from get_hrrr import get_hrrr_wind_10m, get_hrrr_wind_agl

In [ ]:
plantnames = [line.rstrip("\n") for line in open('../targets1')]
plantnames

In [ ]:
CONFIG

In [ ]:
glob.glob(f"{CONFIG['results_folder']}/*")

In [ ]:
loc_name = 'Intermountain'
granules = [k.split('/')[-1][5:-4] for k in glob.glob(f"{CONFIG['results_folder']}/{loc_name}/*_RAD_*")]

exclude = ['EMIT_L1B_RAD_001_20251004T173651_2527711_015', 'EMIT_L1B_RAD_001_20230824T175040_2323612_008',
          'EMIT_L1B_RAD_001_20230601T202821_2315214_013', 'EMIT_L1B_RAD_001_20230206T180123_2303712_003']
granules = [k for k in granules if k not in exclude]
granules

In [ ]:
emit_radfns = [f"{CONFIG['data_folder']}/{loc_name}/{g}.nc" for g in granules]
retr_fns = [f"{CONFIG['results_folder']}/{loc_name}/dSCD_{g}.npy" for g in granules]

In [ ]:
def rgb_stretch(rgb_ds, qlo=2, qhi=98, gamma=2.2, white_background=False):
    da = rgb_ds["radiance"]  # or "reflectance" if you have it

    out = []
    for wl in da["wavelengths"].values:
        b = da.sel(wavelengths=wl)

        # mask nonpositive / invalid
        b = b.where(b > 0)

        lo = b.quantile(qlo/100.0)
        hi = b.quantile(qhi/100.0)

        b = (b - lo) / (hi - lo)
        b = b.clip(0, 1)

        # gamma correction (display gamma)
        b = b ** (1/gamma)

        if white_background:
            b = b.fillna(1.0)
        out.append(b)

    da_out = xr.concat(out, dim="wavelengths")
    da_out = da_out.assign_coords(wavelengths=da["wavelengths"])

    rgb_ds = rgb_ds.copy()
    rgb_ds["radiance"] = da_out
    return rgb_ds

sample_idx = np.random.randint(0, len(emit_radfns))

print(f"Using granule {granules[sample_idx]}")

ds = emit_xarray(emit_radfns[sample_idx], ortho=False)
dSCD = np.load(retr_fns[sample_idx])
ys, xs = np.where(~np.isnan(dSCD))
if len(ys) == 0:
    raise ValueError("No pixels found inside requested box!")

y0, y1 = ys.min(), ys.max()
x0, x1 = xs.min(), xs.max()

rgb = ds.sel(wavelengths=[700, 529, 470], method="nearest")
rgb = rgb_stretch(rgb, qlo=2, qhi=98, gamma=2.2, white_background=True)

rgbimg = rgb["radiance"].transpose("downtrack", "crosstrack", "wavelengths").values[y0:y1+1, x0:x1+1, :]

fig, axs = plt.subplots(1,2,figsize=(20,10))
axs[0].imshow(dSCD[y0:y1+1, x0:x1+1], cmap='RdBu_r')
axs[1].imshow(rgbimg)
axs[1].scatter([len(rgbimg[0])//2], [len(rgbimg)//2], c='red', marker='X')

In [ ]:
loc_info = REFERENCE_PLANTS[loc_name]
ltlon, ltlat = loc_info['LON'], loc_info['LAT']

In [ ]:
import numpy as np
import xarray as xr

def max_center_crop_indices(ds, dSCD, ltlat, ltlon):
    """
    Returns:
      (y0, y1, x0, x1, y_c, x_c)
    where (y_c, x_c) is the pixel closest to (ltlat, ltlon),
    and [y0:y1+1, x0:x1+1] is the largest crop with that pixel at the center.
    """
    lat = ds["lat"].values   # (downtrack, crosstrack)
    lon = ds["lon"].values

    # distance in degrees; good enough for "closest pixel" selection
    # (if you want a bit better: multiply lon diff by cos(lat))
    dlat = lat - ltlat
    dlon = (lon - ltlon) * np.cos(np.deg2rad(ltlat))
    dist2 = dlat*dlat + dlon*dlon

    # ignore invalid geolocation pixels if present
    dist2 = np.where(np.isfinite(dist2), dist2, np.inf)

    y_c, x_c = np.unravel_index(np.argmin(dist2), dist2.shape)

    ny, nx = dSCD.shape  # should match (downtrack, crosstrack)

    # largest symmetric crop around (y_c, x_c)
    half_h = min(y_c, ny - 1 - y_c)
    half_w = min(x_c, nx - 1 - x_c)

    y0, y1 = y_c - half_h, y_c + half_h
    x0, x1 = x_c - half_w, x_c + half_w

    return y0, y1, x0, x1, y_c, x_c


def crop_ds_and_arrays(ds, dSCD, y0, y1, x0, x1):
    ds_crop = ds.isel(downtrack=slice(y0, y1+1), crosstrack=slice(x0, x1+1))
    dSCD_crop = dSCD[y0:y1+1, x0:x1+1]
    return ds_crop, dSCD_crop

y0, y1, x0, x1, yc, xc = max_center_crop_indices(ds, dSCD, ltlat, ltlon)
ds_crop, dSCD_crop = crop_ds_and_arrays(ds, dSCD, y0, y1, x0, x1)

rgb = ds_crop.sel(wavelengths=[700, 529, 470], method="nearest")
rgb = rgb_stretch(rgb, qlo=2, qhi=98, gamma=2.2, white_background=True)

rgbimg = rgb["radiance"].values  # already (downtrack, crosstrack, wavelengths) after your stretch
rgbimg = rgb["radiance"].transpose("downtrack", "crosstrack", "wavelengths").values

fig, axs = plt.subplots(1,2, figsize=(20,10))
axs[0].imshow(dSCD_crop, cmap="RdBu_r")
axs[1].imshow(rgbimg)

axs[0].scatter([dSCD_crop.shape[1]//2], [dSCD_crop.shape[0]//2], c="k", s=60)
axs[1].scatter([rgbimg.shape[1]//2], [rgbimg.shape[0]//2], c="red", marker="X", s=80)

In [ ]:
obs_time = datetime.strptime(granules[sample_idx][-27:-12], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)

geosfp_info = get_geosfp_wind(ltlat, ltlon, obs_time, cache=f'{CONFIG["geosfp"]}/')
geosfp_dir, geosfp_sp = float(geosfp_info["DIR50"]), float(geosfp_info["U50"])

hrrr_agl_info = get_hrrr_wind_agl(ltlat, ltlon, obs_time, layer=(200,600), cache=f'{CONFIG["hrrr"]}')
hrrr_agl_dir, hrrr_agl_sp = float(hrrr_agl_info["dir_from_deg"]), float(hrrr_agl_info["speed_ms"])

hrrr_10m_info = get_hrrr_wind_10m(ltlat, ltlon, obs_time, cache=f'{CONFIG["hrrr"]}')
hrrr_10m_dir, hrrr_10m_sp = float(hrrr_10m_info["dir_from_deg"]), float(hrrr_10m_info["speed_ms"])

print(f"GEOSFP:  \t DIR:{geosfp_dir:0.2f}\t SPD:{geosfp_sp:0.2f}")
print(f"HRRR-AGL:\t DIR:{hrrr_agl_dir:0.2f}\t SPD:{hrrr_agl_sp:0.2f}")
print(f"HRRR-10m:\t DIR:{hrrr_10m_dir:0.2f}\t SPD:{hrrr_10m_sp:0.2f}")

In [ ]:
def wind_to_rotation_deg_in_image(ds, yc, xc, wind_from_deg):
    """
    ds: xarray Dataset with lat/lon coords on (downtrack, crosstrack)
    (yc, xc): center pixel indices in (downtrack, crosstrack)
    wind_from_deg: meteorological direction wind is coming FROM (clockwise from North)
    
    Returns rot_deg suitable for scipy.ndimage.rotate(..., rot_deg, reshape=False)
    so that wind points to +x (right) in the rotated image.
    """
    lat = ds["lat"].values
    lon = ds["lon"].values

    # pick safe finite-difference neighbors
    ny, nx = lat.shape
    y0 = np.clip(yc-1, 0, ny-1); y1 = np.clip(yc+1, 0, ny-1)
    x0 = np.clip(xc-1, 0, nx-1); x1 = np.clip(xc+1, 0, nx-1)

    lat_c = lat[yc, xc]
    if not np.isfinite(lat_c):
        raise ValueError("Center pixel lat is not finite; choose another center or mask invalid geo.")

    # local meters-per-degree factors (good locally)
    m_per_deg_lat = 111_320.0
    m_per_deg_lon = 111_320.0 * np.cos(np.deg2rad(lat_c))

    # image basis vectors expressed in local East/North meters:
    # e_x: one step in crosstrack (to the right)
    dlon_dx = lon[yc, x1] - lon[yc, x0]
    dlat_dx = lat[yc, x1] - lat[yc, x0]
    e_x = np.array([dlon_dx * m_per_deg_lon, dlat_dx * m_per_deg_lat])  # [E, N]

    # e_y: one step in downtrack (downwards)
    dlon_dy = lon[y1, xc] - lon[y0, xc]
    dlat_dy = lat[y1, xc] - lat[y0, xc]
    e_y = np.array([dlon_dy * m_per_deg_lon, dlat_dy * m_per_deg_lat])  # [E, N]

    # Normalize (avoid degenerate cases)
    exn = np.linalg.norm(e_x); eyn = np.linalg.norm(e_y)
    if exn < 1e-12 or eyn < 1e-12:
        raise ValueError("Degenerate local geo gradients; can't define image axes here.")
    e_x /= exn
    e_y /= eyn

    # Wind "to" vector in local East/North
    wind_to = (wind_from_deg + 180.0) % 360.0
    # met convention: 0=N, 90=E
    w_EN = np.array([np.sin(np.deg2rad(wind_to)), np.cos(np.deg2rad(wind_to))])  # [E, N]

    # Project wind onto image axes => components in (x=crosstrack, y=downtrack)
    w_x = np.dot(w_EN, e_x)
    w_y = np.dot(w_EN, e_y)

    # Angle of wind in image coordinates, with +x right, +y down.
    # arctan2 gives angle CCW from +x if y is "up", but our y is "down".
    # So use arctan2(w_y, w_x) and interpret as screen coords.
    angle_deg = np.rad2deg(np.arctan2(w_y, w_x))  # wind pointing at this angle in image plane

    # We want wind to be angle 0 (pointing right). ndimage.rotate is CCW-positive,
    # so rotate by -angle_deg.
    rot_deg = -angle_deg
    return rot_deg, dict(wind_to=wind_to, w_x=w_x, w_y=w_y, angle_deg=angle_deg)

In [ ]:
def rotate_nanaware(a, rot_deg, order=1):
    a = a.astype(np.float32)

    valid = np.isfinite(a).astype(np.float32)
    a0 = np.where(np.isfinite(a), a, 0.0).astype(np.float32)

    # rotate both with SAME settings
    a_rot = rotate(a0, rot_deg, reshape=False, order=order, mode="constant", cval=0.0)
    v_rot = rotate(valid, rot_deg, reshape=False, order=0,   mode="constant", cval=0.0)

    # normalize where we have support
    out = np.full_like(a_rot, np.nan, dtype=np.float32)
    m = v_rot > 0.5
    out[m] = a_rot[m] / v_rot[m]
    return out

# yc, xc should be your chosen center pixel (closest to ltlon/ltlat)
rot_deg, dbg = wind_to_rotation_deg_in_image(ds, yc, xc, wind_from_deg=hrrr_agl_dir)  # or geosfp_dir
dSCD_rot = rotate_nanaware(dSCD_crop, rot_deg, order=1)

rgb_rot = np.stack(
    [rotate(rgbimg[..., k], rot_deg, reshape=False, order=1, mode="constant") for k in range(3)],
    axis=-1
)
print("Rotation:", rot_deg, "deg  | wind_to:", dbg["wind_to"], " | image angle:", dbg["angle_deg"])

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(20,10))
axs[0].imshow(dSCD_rot, cmap="RdBu_r", vmin=-0.03, vmax=0.03)
axs[1].imshow(rgb_rot)

cy, cx = np.array(dSCD_rot.shape)//2
axs[1].arrow(cx-50, cy, 100, 0, width=2, color="white")  # should match downwind
axs[1].scatter([cx],[cy], c="red", marker="X", s=80)
plt.show()

In [ ]:
loc_info = REFERENCE_PLANTS[loc_name]
ltlon, ltlat = loc_info['LON'], loc_info['LAT']
rotated_dscds = []


for granule in granules:
    print(f"Starting {len(rotated_dscds)+1}: {granule}")
    emit_radfn = f"{CONFIG['data_folder']}/{loc_name}/{granule}.nc"
    retr_fn = f"{CONFIG['results_folder']}/{loc_name}/dSCD_{granule}.npy"

    ds = emit_xarray(emit_radfn, ortho=False)
    dSCD = np.load(retr_fn)
    ys, xs = np.where(~np.isnan(dSCD))
    if len(ys) == 0:
        print(f"{granule} FAILED: No pixels found inside requested box!")
        continue
    
    y0, y1 = ys.min(), ys.max()
    x0, x1 = xs.min(), xs.max()

    y0, y1, x0, x1, yc, xc = max_center_crop_indices(ds, dSCD, ltlat, ltlon)
    ds_crop, dSCD_crop = crop_ds_and_arrays(ds, dSCD, y0, y1, x0, x1)

    obs_time = datetime.strptime(granule[-27:-12], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)

    geosfp_info = get_geosfp_wind(ltlat, ltlon, obs_time, cache=f'{CONFIG["geosfp"]}/')
    geosfp_dir, geosfp_sp = float(geosfp_info["DIR50"]), float(geosfp_info["U50"])
    
    hrrr_agl_info = get_hrrr_wind_agl(ltlat, ltlon, obs_time, layer=(200,600), cache=f'{CONFIG["hrrr"]}')
    hrrr_agl_dir, hrrr_agl_sp = float(hrrr_agl_info["dir_from_deg"]), float(hrrr_agl_info["speed_ms"])
    
    hrrr_10m_info = get_hrrr_wind_10m(ltlat, ltlon, obs_time, cache=f'{CONFIG["hrrr"]}')
    hrrr_10m_dir, hrrr_10m_sp = float(hrrr_10m_info["dir_from_deg"]), float(hrrr_10m_info["speed_ms"])

    avg_speed = (geosfp_sp+hrrr_agl_sp+hrrr_10m_sp)/3
    if avg_speed < 1.5:
        print(f"{granule} FAILED: wind speed too low {avg_speed=}")
        continue
    
    rot_deg, dbg = wind_to_rotation_deg_in_image(ds, yc, xc, wind_from_deg=hrrr_agl_dir)  # or geosfp_dir
    dSCD_rot = rotate_nanaware(dSCD_crop, rot_deg, order=1)
    print(f"{granule} SUCCESS! appending")
    rotated_dscds.append(dSCD_rot)

In [ ]:
low_wind = ['EMIT_L1B_RAD_001_20240212T214534_2404314_009', 'EMIT_L1B_RAD_001_20241001T191716_2427513_010', 
            'EMIT_L1B_RAD_001_20230202T193710_2303313_007']

In [ ]:
def center_crop(a, out_h, out_w):
    """Center-crop 2D array a to (out_h, out_w). Returns cropped view."""
    h, w = a.shape
    cy, cx = h // 2, w // 2
    hh, hw = out_h // 2, out_w // 2
    return a[cy - hh: cy + hh + 1, cx - hw: cx + hw + 1]

In [ ]:
for k in rotated_dscds:
    print(k.shape)

In [ ]:
min_pix = 100  # or 100, up to you

rotated_ok = [
    a for a in rotated_dscds
    if (a.shape[0] > min_pix and a.shape[1] > min_pix)
]

print(f"keeping {len(rotated_ok)} / {len(rotated_dscds)} frames")

hs = np.array([a.shape[0] for a in rotated_ok])
ws = np.array([a.shape[1] for a in rotated_ok])

H = int(hs.min())
W = int(ws.min())

# force odd so there is an exact center pixel
H = H if (H % 2 == 1) else (H - 1)
W = W if (W % 2 == 1) else (W - 1)

print("stack size:", H, W)

In [ ]:
stack = np.stack(
    [center_crop(a, H, W) for a in rotated_ok],
    axis=0
)
mean_stack = np.nanmean(stack, axis=0)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14,8), layout='constrained')

v = mean_stack[np.isfinite(mean_stack)]
lim = np.nanpercentile(np.abs(v), 99.5)   # tighter than 99.5
im = axs[0].imshow(mean_stack, cmap='RdBu_r', vmin=-lim, vmax=lim)
plt.colorbar(im, ax=axs[0])
axs[0].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[0].set_title('Raw dSCD')



im = axs[1].imshow(gaussian_filter(mean_stack, 2), cmap="RdBu_r", vmin=-lim, vmax=lim)
plt.colorbar(im, ax=axs[1])
axs[1].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[1].set_title('Gaussian filter')


mask = np.isfinite(mean_stack)
# nearest-neighbor fill
_, idx = distance_transform_edt(~mask, return_indices=True)
filled = mean_stack[tuple(idx)]   # copies nearest valid value into each NaN pixel
tv = denoise_tv_chambolle(filled, weight=0.2)
tv[~mask] = np.nan


im = axs[2].imshow(tv, cmap='RdBu_r', origin='upper', vmin=-0.001, vmax=0.001)
plt.colorbar(im, ax=axs[2])
axs[2].scatter(mean_stack.shape[1] // 2, mean_stack.shape[0] // 2, marker='x', s=50, c='green')
axs[2].set_title('TV filter')

fig.suptitle("Intermountain Power Plant Time Averaged Retrieval", fontsize=20)

In [ ]:
dwer